# N097 · 字典序构造与剩余可行性

**目标：** 兼顾当前更优字符和未来完成约束。

**先修：** N054, N020, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 单调栈；剩余次数；去重标记；前导零；构造可行性。

**配套讲解来源：** [同名逐章意见](../../comment/097_greedy_lexicographic_construction.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章的三个问题都在"构造一个尽可能小的/合法的输出"，共同主线是：**贪心地让当前位置尽量好，但每一步都要先确认"未来还来得及补救"**——这就是标题里"剩余可行性"的含义。

**问题一（移掉 K 位数字，LeetCode 402）**：给一个数字字符串 `num`，删掉恰好 k 位，让剩下的数字尽可能小。例如 `num='1432219', k=3`：**答案是 '1219'**。为什么？删掉 4、3、2（第一个 2）后剩下 1219；任何别的删法（比如删 432、9 或三位 1 开头的组合）都会得到 1432、1219 之外的更大值。最终保留位数固定是 7−3=4 位，位数相同时"字典序最小"就等价于"数值最小"。

**问题二（去重字母，LeetCode 316）**：给字符串 s，要求去掉重复字母使得每个字母只出现一次，且**保持相对顺序**（只能靠删字符得到子序列），返回字典序最小的结果。例如 `s='cbacdcbc'`：**答案是 'acdb'**——四个不同字母 a、b、c、d 各留一个，且留下的顺序必须和原串一致；'acdb' 是所有合法子序列里字典序最小的（比它小的 'ab..'、'acbb' 之类的要么顺序不合法，要么没包含全部字母）。

**问题三（重构字符串，LeetCode 767）**：重排字符串 s 的字符，使得任何两个相邻字符都不相同；不可行返回空串。例如 `s='aab'`：**答案是 'aba'**（a 和 b 各自隔开）。什么时候不可行？某个字母出现次数太多，比如 'aaa'——3 个 a 塞进长度 3 的串里必然相邻。

## 2. 基础知识：先读懂这些词

- **字典序（lexicographic order）**：两个字符串逐位比较，第一处不同的位置决定大小（'1219' < '1432' 因为第 2 位 2<4）。所以"让结果小"的着力点是**最早出现差异的位置**，越靠前的位置越值得优化。
- **单调栈（monotonic stack）**：维护一个栈，让栈内元素保持递增（或递减）；新元素入栈前，把破坏单调性的栈顶元素弹掉。本章两个题都用"递增栈"：一旦发现"左边有个更大的字符"，左边那个就是优化空间，弹它。
- **删除预算（budget k）**：402 题里你只能删 k 次，弹栈一次就是花一次预算。预算花完（k=0）之后就算看到更优机会也不能再弹了，这是"当前更优"和"未来约束"的第一处制衡。
- **剩余次数 / 最后出现位置（last occurrence）**：316 题里每个字母必须被保留恰好一次。想弹掉栈顶字母前，必须检查它**以后还会不会再次出现**（`last[c] > i`）；如果这是它最后一次出场，弹了它就永远失去了这个字母，方案直接不可行。这是"剩余可行性"最典型的体现。
- **去重标记（used 集合）**：一个 `set` 记录"哪些字母已经在栈里"。已经在栈里的字母再次出现时直接跳过——它的副本是多余的，留着只会更长。
- **前导零（leading zeros）**：402 题删完可能得到 '0200' 这种带前导零的串，代表数值 200；代码里 `lstrip('0')` 把前导零剥掉，剥空了就返回 '0'。
- **构造可行性（feasibility）**：767 题的可行性判据：出现次数最多的字母不超过 `(n+1)//2`。直觉是长度 n 的串有 (n+1)//2 个"奇数位"（第 1、3、5…位），同一个字母最多只能占满这些互不相邻的位置。
- **冷却 / 暂存（pending）**：767 题里刚输出过的字符不能立刻再输出，所以把它"暂存"在一边冷却一轮，下一轮再放回堆里。这防止了 'aa' 相邻。
- **频次堆（max-heap by count）**：767 题用堆（存 `(-次数, 字符)`，负号让次数最多的在堆顶）保证每次取当前剩余最多的字符——最多的最紧迫，先安排它才不至于最后挤在一起。

## 3. 思路推导：从小例子开始

**移掉 K 位数字的思路：**

1. **Step 1：** 从左往右扫描，把字符压进栈，维持栈内递增。
2. **Step 2：** 压入字符 c 前，只要"栈顶比 c 大 且 还有删除预算"，就弹掉栈顶（弹一次预算 k 减一）。直觉：让更小的 c 顶替左边更大的字符，最早的差异位变小。
3. **Step 3：** 扫完后预算还没用完（说明整个数已经是递增的，例如 '12345'），那就从末尾删——末尾最大、贡献最小，`del stack[-k:]` 一刀切。
4. **Step 4：** 拼接、剥前导零、空则回 '0'。

手算 `num='1432219', k=3`：读 1 → 栈 [1]；读 4 → 1<4 直接入，栈 [1,4]；读 3 → 4>3 弹 4（k=2），1<3 入，栈 [1,3]；读 2 → 3>2 弹 3（k=1），1<2 入，栈 [1,2]；读 2 → 2>2 不成立，直接入，栈 [1,2,2]；读 1 → 2>1 弹 2（k=0），预算耗尽，后面不能再弹，1 入，栈 [1,2,1]；读 9 → 入，栈 [1,2,1,9]。输出 '1219'。

**去重字母的思路：**

1. **Step 1：** 预处理 `last` 表：每个字母最后一次出现的下标。
2. **Step 2：** 逐字符扫描；已在栈里的字母直接跳过。
3. **Step 3：** 压入 c 前，只要"栈顶比 c 大 且 栈顶字母以后还会出现"（`last[栈顶] > i`），就弹出栈顶并从 used 里注销——用更小的 c 换掉它，反正它后面还有副本能补位。
4. **Step 4：** 压入 c 并登记 used。扫完后的栈就是答案。

手算（“运行示例”部分 的 `s='cbacdcbc'`，`last={c:7,b:6,a:2,d:4}`）：

| 位置 | 读入 | 栈 | 说明 |
|---|---|---|---|
| 0 | c | c | 直接入栈 |
| 1 | b | b | c>b 且 c 以后还有（last[c]=7>1），弹 c 入 b |
| 2 | a | a | b>a 且 b 以后还有（6>2），弹 b 入 a |
| 3 | c | ac | a<c 不弹，直接入 |
| 4 | d | acd | c<d 不弹，直接入 |
| 5 | c | acd | c 已在栈中，跳过 |
| 6 | b | acdb | d>b 但 d 以后**不会**再出现（last[d]=4<6），不敢弹 d，b 入栈 |
| 7 | c | acdb | c 已在栈中，跳过 |

这张表就是 “运行示例”部分 运行后打印的内容（“运行示例”部分 的表没有"说明"列）。位置 6 是全题的点睛之笔：b 明明比栈顶 d 小，却不敢换——因为 d 是最后一次出场，弹了它答案里就没有 d 了。

**重构字符串的思路：**

1. **Step 1：** 统计频次；若最大频次 > (n+1)//2，直接返回空串（不可能成功）。
2. **Step 2：** 所有 (−次数, 字符) 进堆。
3. **Step 3：** 每轮弹出堆顶（当前剩余最多的字符）接到输出末尾；把上一轮"冷却中"的字符（若还有剩余）放回堆；本轮字符扣一次次数后进入冷却。冷却机制保证同一字符不会连续两轮被选中。
4. **Step 4：** 堆空时如果冷却区里还有剩余字符（次数没耗光），说明塞不下了，返回空串；否则返回结果。

手算 `s='aab'`：堆 [(−2,a),(−1,b)]。第 1 轮弹 a，输出 'a'，a 冷却（剩 1 次）；第 2 轮弹 b，输出 'ab'，a 解冻回堆，b 冷却（剩 0 次）；第 3 轮弹 a，输出 'aba'，b 冷却完毕无剩余不入堆。堆空、冷却区干净，返回 'aba'。

## 4. 核心代码：remove_k_digits

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def remove_k_digits(num, k):
    stack = []
    for c in num:
        while k and stack and (stack[-1] > c):
            stack.pop()
            k -= 1
        stack.append(c)
    if k:
        del stack[-k:]
    return ''.join(stack).lstrip('0') or '0'
```

### 逐段读懂代码

“分段实现”部分 先导入 `Counter` 和堆的三个函数，然后是三个独立函数。

**函数一：移掉 K 位数字（以下是 “分段实现”部分 的逐字代码）**

```python
def remove_k_digits(num, k):
    stack = []
    for c in num:
        while k and stack and (stack[-1] > c):
            stack.pop()
            k -= 1
        stack.append(c)
    if k:
        del stack[-k:]
    return ''.join(stack).lstrip('0') or '0'
```

- `while k and stack and stack[-1]>c` 三个条件缺一不可：`k` 是还有预算；`stack` 是栈非空（空栈没有栈顶可比）；`stack[-1]>c` 是栈顶严格大于新字符——相等时不弹（弹不弹结果一样，留着预算后面用更划算）。
- `if k: del stack[-k:]`：扫描结束时预算有剩，说明剩余序列已经递增，最优删法就是砍掉最大的末尾 k 个。
- `''.join(stack).lstrip('0') or '0'`：先拼串再剥前导零；如果剥完是空串（比如 '10' 删 1 位、或 '000' 的情况），`or '0'` 让它落到 '0'——这是 Python 里"空串为假值"特性的紧凑用法。

**函数二：去重字母（以下是 “分段实现”部分 的逐字代码）**

```python
def remove_duplicate_letters(s):
    last = {c: i for i, c in enumerate(s)}
    used = set()
    stack = []
    for i, c in enumerate(s):
        if c in used:
            continue
        while stack and stack[-1] > c and (last[stack[-1]] > i):
            used.remove(stack.pop())
        stack.append(c)
        used.add(c)
    return ''.join(stack)
```

- 字典推导式 `last={c:i for i,c in enumerate(s)}` 从左往右扫一遍，后面出现的下标不断覆盖前面的，最终每个字母映射到它**最后一次**出现的下标——一行完成预处理。
- `if c in used: continue`：字母已在栈中，它的这个副本是多余的，跳过（既不重复入栈，也不触发弹栈）。
- 弹栈条件 `stack and stack[-1]>c and last[stack[-1]]>i` 比函数一多了一道保险：`last[stack[-1]]>i` 确认栈顶字母在位置 i 之后还会出现，弹掉它以后还有机会补回来；否则坚决不弹。弹栈时 `used.remove(...)` 同步注销，保证它后面再次出现时能重新入栈。
- 返回 `''.join(stack)`：栈从底到顶的顺序就是原串的相对顺序，不需要反转。

**函数三：重构字符串（以下是 “分段实现”部分 的逐字代码）**

```python
def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-n, c) for c, n in counts.items()]
    heapify(heap)
    pending = (0, '')
    out = []
    while heap:
        neg, c = heappop(heap)
        out.append(c)
        if pending[0] < 0:
            heappush(heap, pending)
        pending = (neg + 1, c)
    return ''.join(out) if pending[0] == 0 else ''
```

- `max(counts.values(),default=0)`：`default=0` 是给空字符串兜底的——空串没有字符，`max` 会抛异常，给了默认值就返回 0，自然通过可行性检查并输出空串（“正确性与复杂度”部分 的"空输入也返回空串"契约）。
- `heap=[(-n,c) for c,n in counts.items()]`：存负次数让最频繁的字符当堆顶；元组第二位是字符，次数相同时按字符序排，保证行为确定。
- `pending` 是冷却区，初始 `(0,'')` 表示没有字符在冷却。
- 循环体：弹出堆顶接到输出；若冷却区里的字符还有剩余（`pending[0]<0`，即扣掉一次后次数仍为正），放回堆；然后把刚输出的字符扣掉一次（`neg+1`，负数加一就是次数减一）送进冷却。顺序很重要：先放回旧冷却、再登记新冷却，同一轮里刚弹出的字符绝不可能被连续弹出。
- 最后 `if pending[0]==0`：堆空时冷却区也该是干净的；若冷却区还挂着剩余次数，说明有字符没排完（理论上前置检查通过时不会发生，这行是防御性收尾），返回空串。

## 5. 分段实现：按顺序运行

**本章接口：** `remove_k_digits(num, k)`、`remove_duplicate_letters(s)`、`reorganize_string(s)`

### 导入本章使用的库

In [1]:
from collections import Counter
from heapq import heapify, heappush, heappop

### remove_k_digits

In [2]:
def remove_k_digits(num, k):
    stack = []
    for c in num:
        while k and stack and (stack[-1] > c):
            stack.pop()
            k -= 1
        stack.append(c)
    if k:
        del stack[-k:]
    return ''.join(stack).lstrip('0') or '0'

### remove_duplicate_letters

In [3]:
def remove_duplicate_letters(s):
    last = {c: i for i, c in enumerate(s)}
    used = set()
    stack = []
    for i, c in enumerate(s):
        if c in used:
            continue
        while stack and stack[-1] > c and (last[stack[-1]] > i):
            used.remove(stack.pop())
        stack.append(c)
        used.add(c)
    return ''.join(stack)

### reorganize_string

In [4]:
def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-n, c) for c, n in counts.items()]
    heapify(heap)
    pending = (0, '')
    out = []
    while heap:
        neg, c = heappop(heap)
        out.append(c)
        if pending[0] < 0:
            heappush(heap, pending)
        pending = (neg + 1, c)
    return ''.join(out) if pending[0] == 0 else ''

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='cbacdcbc'; last={c:i for i,c in enumerate(s)}; stack=[]; used=set(); rows=[]
for i,c in enumerate(s):
    if c not in used:
        while stack and stack[-1]>c and last[stack[-1]]>i: used.remove(stack.pop())
        stack.append(c); used.add(c)
    rows.append((i,c,''.join(stack)))
show_table(['位置','读入','栈'],rows)

位置,读入,栈
0,c,c
1,b,b
2,a,a
3,c,ac
4,d,acd
5,c,acd
6,b,acdb
7,c,acdb


## 7. 正确性、适用条件与复杂度

单调栈删除发生在能改善最早差异的位置，未用完预算时应删末尾。去重栈的弹出必须有未来副本，才能维持覆盖所有字母的可行性。重排的必要条件是最大频数不超过其余字符数加1；优先安排最多的可选字符并隔离上一字符，可维持该可行性条件。

**代价：** 删除数字与去重字母O(n)时间、O(n)空间；重排O(n log σ)时间、O(σ+n)空间，σ为不同字符数。无可行重排返回空串；空输入也返回空串。

### 展开理解

**402 为什么对？** 字典序由最早的差异位决定。扫描时若栈顶大于新字符 c，那么"用 c 顶替栈顶"能让结果串在更早的位置变小，这是当下能做的最划算改进；预算花完后不能再做这种替换，只能被动接受。若扫完预算有余，说明序列已经递增——递增序列里任何一位都小于等于后一位，删谁都不如删末尾（末尾对高位的伤害最小）。前导零的剥离不改变数值大小，只是把同一个数写成规范形式。

**316 为什么对？** 弹栈的三个条件合起来是一条安全线：栈顶更大（换掉它确实更优）且栈顶以后还会出现（弹掉它不会导致这个字母缺席）。因为每个字母必须恰好保留一次，"以后还有副本"是可行性的底线；反过来说，只要弹出的每个字母后面都还有出场机会，栈里最终就能集齐全部字母且顺序合法（栈序是原串子序列），同时每一步都在让最早差异位变小，所以得到的是字典序最小的合法子序列。

**767 为什么对？** 两句话。第一句（必要性）：若某字符出现 m 次，把它排进长 n 的串且互不相邻，需要 n ≥ 2m−1（每个实例之间至少隔一个别的字符），等价于 m ≤ (n+1)//2；超了就无解，前置检查返回空串。第二句（充分性/维持）：每轮取"剩余次数最多"的字符，拿出后立刻冷却。可以这样理解：只要可行性条件成立，取出最多次数者后，它的次数减一，条件对剩下的串依然成立（最多的都被优先消耗了，差距只会越拉越平），冷却机制又杜绝了相邻重复，所以一路排到底必然成功。

**复杂度。** 402 和 316 都是每个字符最多入栈一次、出栈一次，所以是 O(n) 时间、O(n) 空间。767 的堆大小是不同字符数 σ，每个字符实例消耗一次 O(log σ) 的堆操作，总计 O(n log σ) 时间、O(σ+n) 空间。拿具体数字感受：n=10⁵ 的字符串，402/316 就是二十万次栈操作，毫秒级；767 在 σ=26 时 log σ≈5，也就五十万级操作，同样瞬间完成。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试分两层。

**第一层：三条固定断言。**

- `assert remove_k_digits('1432219',3)=='1219'`：正常值测试，即正文手算的例子，验证"递增栈 + 预算控制"的主流程。
- `assert remove_k_digits('10200',1)=='200'`：特殊值测试，专打**前导零**——删 1 位的最优结果是 '0200'，若不剥前导零就会返回错误形态；剥完 '200' 才是规范答案。
- `assert remove_duplicate_letters('cbacdcbc')=='acdb'`：正常值测试，即 “运行示例”部分 手算的例子，其中位置 6 那步"不敢弹 d"正是关键路径。

**第二层：三组穷举对拍（全部小规模枚举，不是随机）。**

- 重构字符串组：枚举所有长度 1~6 的 'ab' 二元串；用 `max(Counter(s).values())<=(n+1)//2` 算出"理论上可不可行"，断言 `bool(answer)==feasible`（该出空串就出空串，该有解就不出空串）；有解时再验证两条性质：`Counter(answer)==Counter(s)`（字符用量一模一样，没丢没多）和 `all(a!=b for a,b in zip(answer,answer[1:]))`（相邻互异）。
- 去重字母组：枚举所有长度 5 的 'abc' 串（3⁵=243 个）。参照解的构造很巧：正确答案长度一定是不同字母数 k；枚举所有 k 个下标的组合，只要选出的字符恰好覆盖全部 k 种字母（`len({s[i] for i in ids})==k`），拼成一个候选子序列；所有候选取字典序最小就是标准答案。断言本章函数的输出等于这个最小值。
- 删数字组：枚举所有长度 4 的 '012' 串 × k=0..4。参照解：枚举所有"保留 4−k 位"的下标组合拼成串（空拼 `or '0'` 兜底为 '0'），取 `int(...)` 最小。断言用 `int(remove_k_digits(s,k))` 比数值而不是比字符串——这正好绕开前导零的表示差异（'0200' 和 '200' 数值相同），同时也是练习 2 要讨论的"数值最小 vs 字典序最小"的伏笔。

In [6]:
assert remove_k_digits('1432219', 3) == '1219'

assert remove_k_digits('10200', 1) == '200'

assert remove_duplicate_letters('cbacdcbc') == 'acdb'

from itertools import product, combinations

for n in range(1, 7):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        answer = reorganize_string(s)
        feasible = max(Counter(s).values()) <= (n + 1) // 2
        assert bool(answer) == feasible
        if feasible:
            assert Counter(answer) == Counter(s) and all((a != b for a, b in zip(answer, answer[1:])))

for chars in product('abc', repeat=5):
    s = ''.join(chars)
    k = len(set(s))
    options = [''.join((s[i] for i in ids)) for ids in combinations(range(5), k) if len({s[i] for i in ids}) == k]
    assert remove_duplicate_letters(s) == min(options)

for chars in product('012', repeat=4):
    s = ''.join(chars)
    for k in range(5):
        choices = [int(''.join((s[i] for i in ids)) or '0') for ids in combinations(range(4), 4 - k)]
        assert int(remove_k_digits(s, k)) == min(choices)

print('N097: 所有本章断言通过')

N097: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 证明字符以后不再出现时不能弹出。

**练习 2：** 比较数值最小与字典序最小。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（证明"以后不再出现的字符不能弹"）**：思路是把"不可弹"变成反证。提示方向：假设位置 i 处栈顶字母 x 满足 `last[x] < i`（后面不再出现），但你还是把它弹了——那么 x 这个字母从此再无机会入栈，最终栈里没有 x，而题目要求每个不同字母恰好出现一次，方案直接不可行，连"比原来更小"都谈不上。手算示例直接用 “运行示例”部分 位置 6 那一步：'d' 的 last 是 4，位置 6 之后 d 再也不出现，所以 b 只能乖乖叠在 d 后面。你可以做个实验性验证：把代码里 `last[stack[-1]]>i` 这个条件临时删掉跑一遍 'cbacdcbc'，看看输出是不是丢掉了某个字母、断言是不是炸了（记得改完改回来）。

**练习 2（比较数值最小与字典序最小）**：思路是分"剩余长度固定"和"有前导零"两种情况讨论。提示方向：402 题删 k 位后剩余长度固定为 n−k，等长的数字串比较数值和比较字典序**结果一致**（逐位比较，第一位不同就分胜负）——所以算法追求字典序最小等价于追求数值最小。但有前导零时表示会分歧：'0200' 字典序上比 '200' 小（'0'<'2'），数值上却相等；本章代码统一剥成规范形式 '200'，而 “自动测试”部分 的测试用 `int(...)` 比较数值，两边对得上。再想一个边界：k 等于 n 时整串删光，答案规定为 '0' 而不是空串——想想为什么 `or '0'` 那一步不可缺少。验证可以用删数字穷举组的方法：小串上分别求"字典序最小串"和"数值最小的串"，观察剥前导零之后两者何时一致。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter

from heapq import heapify, heappush, heappop

def remove_k_digits(num, k):
    stack = []
    for c in num:
        while k and stack and (stack[-1] > c):
            stack.pop()
            k -= 1
        stack.append(c)
    if k:
        del stack[-k:]
    return ''.join(stack).lstrip('0') or '0'

def remove_duplicate_letters(s):
    last = {c: i for i, c in enumerate(s)}
    used = set()
    stack = []
    for i, c in enumerate(s):
        if c in used:
            continue
        while stack and stack[-1] > c and (last[stack[-1]] > i):
            used.remove(stack.pop())
        stack.append(c)
        used.add(c)
    return ''.join(stack)

def reorganize_string(s):
    counts = Counter(s)
    if max(counts.values(), default=0) > (len(s) + 1) // 2:
        return ''
    heap = [(-n, c) for c, n in counts.items()]
    heapify(heap)
    pending = (0, '')
    out = []
    while heap:
        neg, c = heappop(heap)
        out.append(c)
        if pending[0] < 0:
            heappush(heap, pending)
        pending = (neg + 1, c)
    return ''.join(out) if pending[0] == 0 else ''

# 示例与回归测试
assert remove_k_digits('1432219', 3) == '1219'

assert remove_k_digits('10200', 1) == '200'

assert remove_duplicate_letters('cbacdcbc') == 'acdb'

from itertools import product, combinations

for n in range(1, 7):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        answer = reorganize_string(s)
        feasible = max(Counter(s).values()) <= (n + 1) // 2
        assert bool(answer) == feasible
        if feasible:
            assert Counter(answer) == Counter(s) and all((a != b for a, b in zip(answer, answer[1:])))

for chars in product('abc', repeat=5):
    s = ''.join(chars)
    k = len(set(s))
    options = [''.join((s[i] for i in ids)) for ids in combinations(range(5), k) if len({s[i] for i in ids}) == k]
    assert remove_duplicate_letters(s) == min(options)

for chars in product('012', repeat=4):
    s = ''.join(chars)
    for k in range(5):
        choices = [int(''.join((s[i] for i in ids)) or '0') for ids in combinations(range(4), 4 - k)]
        assert int(remove_k_digits(s, k)) == min(choices)

print('N097: 所有本章断言通过')

N097: 所有本章断言通过


## 11. 代表题与迁移

- **402. Remove K Digits（canonical）**：这题就是 `remove_k_digits` 的原题。它练的是**单调栈 + 删除预算**：每一步的"更优替换"都受剩余预算约束。
- **316. Remove Duplicate Letters（canonical）**：这题就是 `remove_duplicate_letters` 的原题。它练的是单调栈加**剩余可行性检查**（last 表）：敢于反悔的前提是反悔不毁掉未来。
- **767. Reorganize String（extension）**：这题是 `reorganize_string` 的原题（官方标 median 难度，本章把它当作"构造可行性"的拓展练习）。它练的是**频次堆 + 冷却暂存**的构造套路，以及 `(n+1)//2` 这个可行性判据的直觉。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。